# Voice Shield · 01 · Baseline on In-the-Wild

Scores the public **XLS-R + SLS** checkpoint on the In-the-Wild dataset with **no training**.
The checkpoint's author reports **7.84% EER** on In-the-Wild, so our number should land close to that.
If it does, our model port is correct and we can start fine-tuning.

**Before you press Run All**
1. Settings → Accelerator → **GPU T4 x2** (or P100).
2. Settings → **Internet on** (needs a phone-verified Kaggle account). The checkpoint downloads from HuggingFace.
3. Add Input → search **"in the wild audio deepfake"** and add a copy that contains `meta.csv` and the wav files.

All our code is inside this notebook (the `%%writefile` cells), so nothing else needs uploading.

It scores three ways:
- `paper`: exactly like the original code (first 4 s, silence kept). Should give about 7.84%.
- `trimmed`: silence cut at start and end, like our app does.
- `phone`: trimmed and then made to sound like a phone call (8 kHz, telephone band, G.711 μ-law).

## Our code
Run these cells as they are. Each one writes one file of our `vs` package.

In [ ]:
import os
os.makedirs('vs', exist_ok=True)

In [ ]:
%%writefile vs/__init__.py
"""Voice Shield: AI-voice and scam-call detection."""


In [ ]:
%%writefile vs/audio.py
"""Audio helpers: loading, clean-up, and phone-call simulation.

Everything works on mono float32 numpy arrays in the range [-1, 1].
The detector expects 16 kHz audio; phone audio (Twilio) is 8 kHz mu-law.
"""

from __future__ import annotations

import shutil
import subprocess
import tempfile
from math import gcd
from pathlib import Path

import numpy as np
import soundfile as sf
from scipy.signal import butter, resample_poly, sosfilt

SR = 16000            # detector sample rate
PHONE_SR = 8000       # telephone sample rate
WINDOW = 64600        # samples the SLS model was trained on (~4.04 s at 16 kHz)


# ---------------------------------------------------------------- loading

def resample(x: np.ndarray, sr_in: int, sr_out: int) -> np.ndarray:
    if sr_in == sr_out:
        return x.astype(np.float32)
    g = gcd(sr_in, sr_out)
    return resample_poly(x, sr_out // g, sr_in // g).astype(np.float32)


def load_audio(path: str | Path, sr: int = SR) -> np.ndarray:
    """Load any file libsndfile can read (wav, flac, ogg, mp3) as mono at `sr`."""
    x, file_sr = sf.read(str(path), dtype="float32", always_2d=True)
    return resample(x.mean(axis=1), file_sr, sr)


def to_mono_float(raw: bytes, sr_in: int, sr: int = SR) -> np.ndarray:
    """Decode an uploaded file's bytes."""
    with tempfile.NamedTemporaryFile(suffix=".audio", delete=False) as f:
        f.write(raw)
        tmp = f.name
    try:
        return load_audio(tmp, sr)
    finally:
        Path(tmp).unlink(missing_ok=True)


# ---------------------------------------------------------------- clean-up

def normalize_loudness(x: np.ndarray, target_dbfs: float = -23.0) -> np.ndarray:
    """Scale to a fixed RMS level so loudness can't be used as a shortcut."""
    rms = float(np.sqrt(np.mean(x ** 2)) + 1e-9)
    y = x * (10 ** (target_dbfs / 20) / rms)
    return np.clip(y, -1.0, 1.0).astype(np.float32)


def trim_silence(x: np.ndarray, sr: int = SR, top_db: float = 35.0,
                 frame_ms: float = 25.0) -> np.ndarray:
    """Remove silence at the start and end (energy based).

    Silence length alone predicts real vs fake in ASVspoof 2019, so we cut it.
    """
    frame = int(sr * frame_ms / 1000)
    if len(x) < frame * 2:
        return x
    n = len(x) // frame
    energy = np.array([np.mean(x[i * frame:(i + 1) * frame] ** 2) for i in range(n)])
    db = 10 * np.log10(energy + 1e-12)
    keep = np.where(db > db.max() - top_db)[0]
    if len(keep) == 0:
        return x
    return x[keep[0] * frame:(keep[-1] + 1) * frame]


def repeat_pad(x: np.ndarray, length: int = WINDOW) -> np.ndarray:
    """Cut or repeat-pad to `length` samples (same as the SLS training code)."""
    if len(x) >= length:
        return x[:length]
    reps = length // max(len(x), 1) + 1
    return np.tile(x, reps)[:length]


def windows(x: np.ndarray, length: int = WINDOW, hop: int = SR) -> list[np.ndarray]:
    """Split a long clip into overlapping model-sized windows."""
    if len(x) <= length:
        return [repeat_pad(x, length)]
    starts = list(range(0, len(x) - length + 1, hop))
    if starts[-1] + length < len(x):
        starts.append(len(x) - length)
    return [x[s:s + length] for s in starts]


# ---------------------------------------------------------------- G.711 mu-law
# Twilio Media Streams always send 8 kHz G.711 mu-law, mono.

def _build_g711_table() -> np.ndarray:
    table = np.zeros(256, dtype=np.float32)
    for i in range(256):
        u = ~i & 0xFF
        sign = u & 0x80
        exponent = (u >> 4) & 0x07
        mantissa = u & 0x0F
        sample = ((mantissa << 3) + 0x84) << exponent
        sample -= 0x84
        table[i] = (-sample if sign else sample) / 32768.0
    return table


_G711 = _build_g711_table()


def g711_decode(b: bytes) -> np.ndarray:
    return _G711[np.frombuffer(b, dtype=np.uint8)]


def g711_encode(x: np.ndarray) -> bytes:
    pcm = np.clip(x * 32768.0, -32635, 32635).astype(np.int32)
    sign = np.where(pcm < 0, 0x80, 0)
    mag = np.abs(pcm) + 0x84
    exponent = np.clip(np.floor(np.log2(mag)).astype(np.int32) - 7, 0, 7)
    mantissa = (mag >> (exponent + 3)) & 0x0F
    return (~(sign | (exponent << 4) | mantissa) & 0xFF).astype(np.uint8).tobytes()


# ---------------------------------------------------------------- phone simulation

_BANDPASS = butter(4, [300, 3400], btype="bandpass", fs=PHONE_SR, output="sos")


def _ffmpeg_codec(x8k: np.ndarray, codec: str) -> np.ndarray | None:
    """Round-trip 8 kHz audio through a real phone codec using ffmpeg, if present."""
    exe = shutil.which("ffmpeg")
    if exe is None:
        return None
    fmt = {"amr": ("libopencore_amrnb", "amr", ["-b:a", "12.2k"]),
           "gsm": ("libgsm", "gsm", [])}[codec]
    with tempfile.TemporaryDirectory() as d:
        src, enc, out = Path(d, "in.wav"), Path(d, f"enc.{fmt[1]}"), Path(d, "out.wav")
        sf.write(src, x8k, PHONE_SR)
        ok = subprocess.run([exe, "-y", "-loglevel", "error", "-i", str(src), "-ar", "8000",
                             "-ac", "1", "-c:a", fmt[0], *fmt[2], str(enc)],
                            capture_output=True).returncode == 0
        if not ok:
            return None
        subprocess.run([exe, "-y", "-loglevel", "error", "-i", str(enc), "-ar", "8000",
                        str(out)], capture_output=True)
        y, _ = sf.read(out, dtype="float32")
        return y[:len(x8k)]


def phone_simulate(x: np.ndarray, sr: int = SR, codec: str | None = None,
                   noise_snr_db: float | None = None,
                   rng: np.random.Generator | None = None,
                   bandpass: bool = True) -> np.ndarray:
    """Make clean audio sound like a phone call, then return it at `sr`.

    Steps: downsample to 8 kHz, telephone band (300-3400 Hz), optional real codec
    (AMR-NB or GSM via ffmpeg), optional background noise, G.711 mu-law (what
    Twilio delivers), back up to `sr`.
    """
    rng = rng or np.random.default_rng()
    y = resample(x, sr, PHONE_SR)
    if bandpass:
        y = sosfilt(_BANDPASS, y).astype(np.float32)
    if codec:
        coded = _ffmpeg_codec(y, codec)
        if coded is not None:
            y = coded
    if noise_snr_db is not None:
        p = np.mean(y ** 2) + 1e-12
        y = y + rng.normal(0, np.sqrt(p / 10 ** (noise_snr_db / 10)), len(y)).astype(np.float32)
    y = g711_decode(g711_encode(np.clip(y, -1, 1)))
    return resample(y, PHONE_SR, sr)


def has_ffmpeg_codecs() -> bool:
    return shutil.which("ffmpeg") is not None


In [ ]:
%%writefile vs/metrics.py
"""Equal Error Rate and related numbers for the results table."""

from __future__ import annotations

import numpy as np


def compute_eer(p_fake_real: np.ndarray, p_fake_fake: np.ndarray) -> tuple[float, float]:
    """EER and the threshold where it happens.

    p_fake_real: fake-probabilities given to real (bonafide) clips
    p_fake_fake: fake-probabilities given to AI (spoof) clips
    Returns (eer as a fraction, threshold on p_fake).
    """
    scores = np.concatenate([p_fake_real, p_fake_fake])
    labels = np.concatenate([np.zeros(len(p_fake_real)), np.ones(len(p_fake_fake))])
    order = np.argsort(scores)
    scores, labels = scores[order], labels[order]
    n_real, n_fake = len(p_fake_real), len(p_fake_fake)
    # threshold just above scores[i]: everything <= scores[i] is called real
    false_alarm = 1 - np.cumsum(labels == 0) / n_real     # real clips called fake
    miss = np.cumsum(labels == 1) / n_fake                # fake clips called real
    i = int(np.argmin(np.abs(false_alarm - miss)))
    return float((false_alarm[i] + miss[i]) / 2), float(scores[i])


def rates_at(p_fake_real: np.ndarray, p_fake_fake: np.ndarray, threshold: float) -> dict:
    """False-alarm and miss rates if we call 'fake' when p_fake > threshold."""
    return {
        "false_alarm_on_real": float(np.mean(p_fake_real > threshold)),
        "missed_fakes": float(np.mean(p_fake_fake <= threshold)),
    }


In [ ]:
%%writefile vs/model.py
"""XLS-R + SLS deepfake detector, rebuilt on HuggingFace transformers.

The public checkpoint (Zhang et al. 2024, reproduced by Sukhdeve) was trained
with fairseq, which is hard to install on Windows and new Pythons. This file
rebuilds the same network with `transformers.Wav2Vec2Model` and converts the
fairseq weights on load, so the laptop and Kaggle run the same code.

Model input: (batch, 64600) raw 16 kHz waveform, no normalisation.
Model output: log-probabilities for [spoof, bonafide]  (index 1 = real voice).
"""

from __future__ import annotations

import re

import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import Wav2Vec2Config, Wav2Vec2Model

XLSR_NAME = "facebook/wav2vec2-xls-r-300m"
CKPT_REPO = "sukhdeveyash/XLS-R-SLS-Deepfake-Detection"
CKPT_FILE = "v1/epoch_2.pth"   # the repo's recommended checkpoint (7.84% EER on In-the-Wild)


def xlsr_config() -> Wav2Vec2Config:
    cfg = Wav2Vec2Config.from_pretrained(XLSR_NAME)
    # fairseq was called with mask=False and every layer must run, so turn off
    # SpecAugment masking and LayerDrop.
    cfg.apply_spec_augment = False
    cfg.mask_time_prob = 0.0
    cfg.mask_feature_prob = 0.0
    cfg.layerdrop = 0.0
    return cfg


class SLSModel(nn.Module):
    """XLS-R 300M -> Selective Layer Summarization -> small classifier head."""

    def __init__(self, cfg: Wav2Vec2Config | None = None):
        super().__init__()
        self.ssl = Wav2Vec2Model(cfg or xlsr_config())
        self.first_bn = nn.BatchNorm2d(num_features=1)
        self.selu = nn.SELU(inplace=True)
        self.fc0 = nn.Linear(1024, 1)
        self.fc1 = nn.Linear(22847, 1024)   # 341 * 67 after 3x3 max-pool of (201 frames, 1024 dims)
        self.fc3 = nn.Linear(1024, 2)

        # fairseq's layer_results are the raw outputs of all 24 transformer
        # layers. HF applies a final LayerNorm to its last hidden state, so we
        # hook the layers directly to get exactly the same tensors.
        self._layer_out: list[torch.Tensor] = []
        for layer in self.ssl.encoder.layers:
            layer.register_forward_hook(self._grab)

    def _grab(self, _module, _inp, out):
        self._layer_out.append(out[0] if isinstance(out, tuple) else out)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        self._layer_out = []
        self.ssl(x)
        layers = self._layer_out                       # 24 x (B, T, 1024)
        self._layer_out = []

        pooled = torch.stack([h.mean(dim=1) for h in layers], dim=1)   # (B, 24, 1024)
        full = torch.stack(layers, dim=1)                              # (B, 24, T, 1024)
        weight = torch.sigmoid(self.fc0(pooled)).unsqueeze(-1)          # (B, 24, 1, 1)
        feat = (full * weight).sum(dim=1).unsqueeze(1)                 # (B, 1, T, 1024)

        x = self.selu(self.first_bn(feat))
        x = F.max_pool2d(x, (3, 3))
        x = torch.flatten(x, 1)
        x = self.selu(self.fc1(x))
        x = self.selu(self.fc3(x))
        return F.log_softmax(x, dim=1)


# ---------------------------------------------------------------- fairseq -> HF weights

_RULES = [
    (r"feature_extractor\.conv_layers\.(\d+)\.0\.", r"feature_extractor.conv_layers.\1.conv."),
    (r"feature_extractor\.conv_layers\.(\d+)\.2\.1\.", r"feature_extractor.conv_layers.\1.layer_norm."),
    (r"^layer_norm\.", "feature_projection.layer_norm."),
    (r"^post_extract_proj\.", "feature_projection.projection."),
    (r"encoder\.layers\.(\d+)\.self_attn\.", r"encoder.layers.\1.attention."),
    (r"encoder\.layers\.(\d+)\.self_attn_layer_norm\.", r"encoder.layers.\1.layer_norm."),
    (r"encoder\.layers\.(\d+)\.fc1\.", r"encoder.layers.\1.feed_forward.intermediate_dense."),
    (r"encoder\.layers\.(\d+)\.fc2\.", r"encoder.layers.\1.feed_forward.output_dense."),
]
# pretraining-only parts; mask_emb is unused because masking is off
_SKIP = ("quantizer.", "project_q.", "final_proj.", "label_embs", "target_glu.", "mask_emb")


def _pos_conv_keys(hf_keys: set[str]) -> tuple[str, str]:
    base = "ssl.encoder.pos_conv_embed.conv."
    if base + "parametrizations.weight.original0" in hf_keys:
        return base + "parametrizations.weight.original0", base + "parametrizations.weight.original1"
    return base + "weight_g", base + "weight_v"


def convert_fairseq_checkpoint(sd: dict, model: SLSModel) -> dict:
    """Rename a fairseq SLS state dict so it loads into SLSModel."""
    hf_keys = set(model.state_dict().keys())
    g_key, v_key = _pos_conv_keys(hf_keys)
    out = {}
    for k, v in sd.items():
        k = k.removeprefix("module.")
        if not k.startswith("ssl_model.model."):
            out[k] = v                          # classifier head: same names
            continue
        k = k.removeprefix("ssl_model.model.")
        if k.startswith(_SKIP):
            continue
        if k.startswith("encoder.pos_conv.0."):
            tail = k.removeprefix("encoder.pos_conv.0.")
            target = {"weight_g": g_key, "weight_v": v_key,
                      "parametrizations.weight.original0": g_key,
                      "parametrizations.weight.original1": v_key,
                      "bias": "ssl.encoder.pos_conv_embed.conv.bias"}[tail]
            out[target] = v
            continue
        for pat, rep in _RULES:
            k = re.sub(pat, rep, k)
        out["ssl." + k] = v
    return out


def load_sls(checkpoint_path: str, device: str = "cpu") -> SLSModel:
    model = SLSModel()
    raw = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    if isinstance(raw, dict) and "state_dict" in raw:
        raw = raw["state_dict"]
    sd = convert_fairseq_checkpoint(raw, model) if any(
        k.removeprefix("module.").startswith("ssl_model.") for k in raw) else raw
    missing, unexpected = model.load_state_dict(sd, strict=False)
    missing = [k for k in missing if not k.endswith("num_batches_tracked")]
    if missing or unexpected:
        raise RuntimeError(f"Checkpoint did not match the model.\n"
                           f"Missing ({len(missing)}): {missing[:10]}\n"
                           f"Unexpected ({len(unexpected)}): {unexpected[:10]}")
    return model.to(device).eval()


def download_checkpoint(filename: str = CKPT_FILE, cache_dir: str | None = None) -> str:
    from huggingface_hub import hf_hub_download
    return hf_hub_download(repo_id=CKPT_REPO, filename=filename, cache_dir=cache_dir)


In [ ]:
# Settings you can change
SUBSET = None        # e.g. 3000 for a quick test run, None for all ~31.8k clips
BATCH = 32
MODES = ["paper", "trimmed", "phone"]

In [ ]:
import sys, glob, os, time
sys.path.insert(0, os.getcwd())     # the vs/ package written by the cells above

meta = glob.glob('/kaggle/input/**/meta.csv', recursive=True)
assert meta, "Add an In-the-Wild dataset that has meta.csv."
META = meta[0]; WAV_DIR = os.path.dirname(META)
print("In-the-Wild meta:", META)

In [ ]:
import numpy as np, pandas as pd, torch
from torch.utils.data import Dataset, DataLoader
from vs import audio
from vs.model import load_sls, download_checkpoint
from vs.metrics import compute_eer

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device, torch.cuda.get_device_name(0) if device == "cuda" else "")
ckpt = download_checkpoint()
model = load_sls(ckpt, device)          # raises if the weight conversion does not match
print("Checkpoint loaded:", ckpt)

In [ ]:
df = pd.read_csv(META)
df["is_fake"] = (df["label"].str.lower() == "spoof").astype(int)
if SUBSET:
    df = df.groupby("is_fake", group_keys=False).apply(lambda g: g.sample(min(len(g), SUBSET // 2), random_state=0))
print(df["label"].value_counts())

class Clips(Dataset):
    def __init__(self, df, mode):
        self.df, self.mode = df.reset_index(drop=True), mode
    def __len__(self):
        return len(self.df)
    def __getitem__(self, i):
        x = audio.load_audio(os.path.join(WAV_DIR, self.df.file[i]))
        if self.mode in ("trimmed", "phone"):
            x = audio.trim_silence(x)
        if self.mode == "phone":
            x = audio.phone_simulate(x, rng=np.random.default_rng(i))
        x = audio.repeat_pad(x)
        if self.mode != "paper":
            x = audio.normalize_loudness(x)     # the app sets every window to -23 dBFS
        return torch.from_numpy(x), i

In [ ]:
results, all_scores = [], {}
for mode in MODES:
    dl = DataLoader(Clips(df, mode), batch_size=BATCH, num_workers=4)
    p_fake = np.zeros(len(df)); t0 = time.time()
    with torch.inference_mode(), torch.autocast(device, dtype=torch.float16, enabled=device == "cuda"):
        for x, idx in dl:
            p_real = model(x.to(device)).float()[:, 1].exp()
            p_fake[idx.numpy()] = (1 - p_real).cpu().numpy()
    all_scores[mode] = p_fake
    y = df["is_fake"].values
    eer, thr = compute_eer(p_fake[y == 0], p_fake[y == 1])
    fa = float(np.mean(p_fake[y == 0] > 0.5))
    results.append({"mode": mode, "EER %": round(eer * 100, 2), "threshold": round(thr, 4),
                    "false alarms on real @0.5 %": round(fa * 100, 1), "minutes": round((time.time() - t0) / 60, 1)})
    print(results[-1])

res = pd.DataFrame(results); print(res.to_string(index=False))

In [ ]:
out = df[["file", "label"]].copy()
for m, s in all_scores.items():
    out[f"p_fake_{m}"] = np.round(s, 5)
out.to_csv("/kaggle/working/itw_baseline_scores.csv", index=False)
res.to_csv("/kaggle/working/itw_baseline_results.csv", index=False)
print("Saved. Copy the results table above into the chat.")